In [ ]:
from _plot_utils import plot_prototype_similarity_heatmap, plot_prototype_label_indicators

prototype similarity heatmap:
* preprojection learned prototypes
* label-guided projections onto PTB-XL
* label-free projections onto EchoNext

prototype label indictors:
* label-guided projections onto PTB-XL
* label-free projections onto EchoNext

In [ ]:
for label_group in [1, 3, 4]:
    print(f"================ Cat {label_group} ================")
    plot_prototype_similarity_heatmap(
        ckpt_path=f"../external/bbj-lab-protoecgnet/ptbxl-preprojection-checkpoints/cat{label_group}.ckpt",
        title=f"Cat {label_group} Learned Prototypes",
        label_set="ptbxl",
        label_group=label_group,
        save_path=f"figs/cat{label_group}_learned_prototype_similarity_heatmap.png",
    )
    plot_prototype_similarity_heatmap(
        ckpt_path=f"../external/bbj-lab-protoecgnet/ptbxl-classifier-checkpoints/cat{label_group}.ckpt",
        title=f"Cat {label_group} PTB-XL Projected Prototypes",
        label_set="ptbxl",
        label_group=label_group,
        save_path=f"figs/cat{label_group}_ptbxl_prototype_similarity_heatmap.png",
    )
    plot_prototype_similarity_heatmap(
        ckpt_path=f"../outputs/runs/protoecgnet-reproj-cat{label_group}/checkpoints/proj_cat{label_group}/proj_cat{label_group}_projection.pth",
        title=f"Cat {label_group} EchoNext Projected Prototypes",
        label_set="echonext",
        label_group=label_group,
        save_path=f"figs/cat{label_group}_echonext_prototype_similarity_heatmap.png",
    )
    plot_prototype_label_indicators(
        metadata_path=f"../external/bbj-lab-protoecgnet/ptbxl-classifier-checkpoints/cat{label_group}_metadata.json",
        title=f"Cat {label_group} PTB-XL Projected Prototype Labels",
        label_set="ptbxl",
        label_group=label_group,
        save_path=f"figs/cat{label_group}_ptbxl_prototype_label_indicators.png",
    )
    plot_prototype_label_indicators(
        metadata_path=f"../outputs/runs/protoecgnet-reproj-cat{label_group}/checkpoints/proj_cat{label_group}/proj_cat{label_group}_prototype_metadata.json",
        title=f"Cat {label_group} EchoNext Projected Prototype Labels",
        label_set="echonext",
        label_group=label_group,
        save_path=f"figs/cat{label_group}_echonext_prototype_label_indicators.png",
    )

In [ ]:
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics.pairwise import cosine_similarity
from pass_pclr.defines import ECHONEXT_TARGETS

In [ ]:
lp = torch.load("../outputs/runs/pass-pretrain-ptbxl/learn-prototypes/latest/best.ckpt", map_location="cpu")
pip = torch.load("../outputs/runs/pass-pretrain-ptbxl/project-prototypes/latest/proj.ckpt", map_location="cpu")
pit = torch.load("../outputs/runs/pass-ptbxl-to-echonext-w-proj/project-prototypes/latest/proj.ckpt", map_location="cpu")

In [ ]:
def plot_prototype_sims(prototypes, title, tick_sep, save_path):
    sims = cosine_similarity(prototypes)
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(sims, ax=ax)
    ax.set_title(f"{title} (Avg Cos Sim = {sims.mean():0.3f})")
    ax.set_xlabel("Prototype Index")
    ax.set_ylabel("Prototype Index")
    n_prototypes = sims.shape[0]
    ticks = list(range(0, n_prototypes + 1, tick_sep))
    ax.set_xticks(ticks=ticks, labels=ticks)  # type: ignore
    ax.set_yticks(ticks=ticks)
    ax.set_yticklabels(labels=ticks, rotation=0)  # type: ignore
    if save_path is not None:
        fig.tight_layout()
        fig.savefig(save_path)

In [ ]:
plot_prototype_sims(lp["state_dict"]["model.encoder.prototypes"], "PASS-PCLR Learned Prototypes", 8, "figs/pass-pclr-learned-prototypes.png")

In [ ]:
plot_prototype_sims(pip["state_dict"]["model.encoder.prototypes"], "PASS-PCLR Pretrain Projected Prototypes", 8, "figs/pass-pclr-pretrain-projected-prototypes.png")

In [ ]:
plot_prototype_sims(pit["state_dict"]["model.encoder.prototypes"], "PASS-PCLR Transfer Projected Prototypes", 8, "figs/pass-pclr-transfer-projected-prototypes.png")

In [ ]:
# ptbxl = pd.read_csv("/opt/gpudata/ecg/ptb-xl/ptbxl_database.csv")
# pip_ids = pd.read_csv("../outputs/runs/pass-pretrain-ptbxl/project-prototypes/latest/projection_metadata.csv")

In [ ]:
echonext = pd.read_csv("/opt/gpudata/ecg/echonext/EchoNext_metadata_100k.csv")
pit_ids = pd.read_csv("../outputs/runs/pass-ptbxl-to-echonext-w-proj/project-prototypes/latest/projection_metadata.csv")

assert not echonext["ecg_key"].duplicated().any()
echonext = echonext.set_index("ecg_key")

echonext_labels = list(ECHONEXT_TARGETS.keys())
echonext = echonext.rename(columns={v: k for k, v in ECHONEXT_TARGETS.items()})

echonext_indicators = echonext.loc[pit_ids["ecg_id"].to_numpy(), echonext_labels].to_numpy()
echonext_nunique = pit_ids["ecg_id"].nunique()

In [ ]:
def plot_prototype_labels(
    *,  # enforce kwargs
    indicators: np.ndarray,
    n_unique: int,
    labels: list[str],
    title: str,
    tick_sep: int,
    save_path: str | None = None,
):
    n_prototypes, n_labels = indicators.shape
    assert len(labels) == n_labels
    fig, ax = plt.subplots(figsize=(10, 5))
    sns.heatmap(indicators.T, ax=ax, cbar=False, cmap="viridis")
    ax.grid(axis="y")
    xticks = list(range(0, n_prototypes + 1, tick_sep))
    ax.set_xticks(ticks=xticks, labels=xticks)  # type: ignore
    ax.set_yticks(range(0, n_labels + 1))
    ax.set_yticks([x + 0.5 for x in range(0, n_labels)], minor=True)
    ax.set_yticklabels(labels, minor=True)
    ax.tick_params(
        axis="y",
        which="both",
        length=0,
    )
    ax.set_xlabel("Prototype Index")
    ax.set_title(f"{title} (N Unique Samples = {n_unique})")
    if save_path is not None:
        fig.tight_layout()
        fig.savefig(save_path)


In [ ]:
plot_prototype_labels(
    indicators=echonext_indicators,
    n_unique=echonext_nunique,
    labels=echonext_labels,
    title="PASS-PCLR Transfer Projected Prototypes",
    tick_sep=8,
    save_path="figs/pass-pclr-transfer-projected-labels.png",
)